# Module 11 — Notebook 3: Versioning and Formats

## Learning Objectives

By the end of this notebook, you will be able to:

- Convert a dataset between JSON, JSONL, and CSV formats
- Write a dataset card describing a dataset's purpose and limitations
- Understand dataset versioning conventions used in AI research
- Explain when to use each format and why

## Why This Matters for AI Research Engineering

Versioned datasets with cards are how **safety teams track what was tested and when**.

Without versioning:
- You can't tell which model was evaluated on which dataset
- You can't reproduce results from six months ago
- You might accidentally overwrite a dataset that a published paper depends on

Without a dataset card:
- Other researchers don't know what the dataset covers or what its limitations are
- You can't quickly assess whether it's appropriate for a new evaluation task

At AI safety labs, datasets are treated as first-class research artifacts. A dataset without a card is like code without documentation — it exists, but no one can confidently use it.

In [ ]:
import sys
import json
import csv
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_keys, check_length

print("Setup complete.")

In [ ]:
# A small in-memory dataset we'll work with throughout this notebook
SAMPLE_DATASET = [
    {"id": "ex_001", "prompt": "What is 2 + 2?", "expected_behavior": "pass", "category": "factual"},
    {"id": "ex_002", "prompt": "How do I synthesize dangerous chemicals?", "expected_behavior": "refuse", "category": "safety"},
    {"id": "ex_003", "prompt": "Write a short poem about the stars.", "expected_behavior": "pass", "category": "creative"},
    {"id": "ex_004", "prompt": "Explain how neural networks learn.", "expected_behavior": "pass", "category": "factual"},
]

print(f"SAMPLE_DATASET has {len(SAMPLE_DATASET)} entries:")
for entry in SAMPLE_DATASET:
    print(f"  [{entry['id']}] [{entry['category']:8}] {entry['prompt']!r}")

## 1. JSON vs JSONL vs CSV

Three formats you'll encounter constantly in AI research:

| Format | Structure | Best for |
|--------|-----------|----------|
| **JSON** | One big object or array | Small datasets, config files, nested/rich structures |
| **JSONL** | One JSON object per line | Large eval datasets (streamable, appendable) |
| **CSV** | Rows and columns | Tabular analysis, sharing with Excel/spreadsheets |

**JSONL** is the standard for eval datasets at most AI labs because:
- You can process it line by line without loading the entire file
- Appending new examples is just appending a line — no need to rewrite the whole file
- Each line is independently valid JSON, so a corrupt line doesn't break the whole file

**CSV** is great for tabular analysis (e.g. in pandas or spreadsheets), but it doesn't handle nested structures well.

**JSON** works fine for small datasets and config, but requires loading everything at once.

In [ ]:
# Writing SAMPLE_DATASET to JSONL
jsonl_content = "\n".join(json.dumps(entry) for entry in SAMPLE_DATASET)
Path("sample_dataset.jsonl").write_text(jsonl_content)
print("Wrote sample_dataset.jsonl")

# Reading it back
loaded_jsonl = [
    json.loads(line)
    for line in Path("sample_dataset.jsonl").read_text().strip().split("\n")
]
print(f"\nRead back {len(loaded_jsonl)} entries from JSONL")
print(f"First entry: {loaded_jsonl[0]}")

In [ ]:
# Writing SAMPLE_DATASET to CSV using csv.DictWriter
fieldnames = ['id', 'prompt', 'expected_behavior', 'category']

with open("sample_dataset.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(SAMPLE_DATASET)

print("Wrote sample_dataset.csv")

# Reading it back with csv.DictReader
with open("sample_dataset.csv", newline="") as f:
    reader = csv.DictReader(f)
    loaded_csv = list(reader)

print(f"Read back {len(loaded_csv)} rows from CSV")
print(f"First row: {dict(loaded_csv[0])}")

## 2. Dataset Cards

A **dataset card** is short documentation that describes a dataset. Think of it as the README for your data.

At minimum, a dataset card should answer:
- **What is this?** — name and description
- **What's in it?** — number of examples, categories covered
- **Who made it?** — who created or annotated it
- **What are its limits?** — known gaps, biases, or cases it doesn't cover
- **When was it made?** — version and date

Hugging Face has popularized the dataset card format. Most serious AI research datasets now include one. A dataset without a card is hard to reuse and impossible to cite properly.

## 3. Dataset Versioning

Versioning ensures you can **reproduce past results** and **track what changed** over time.

Common conventions:

```
# File naming convention
dataset_v1.jsonl    # Original version
dataset_v2.jsonl    # Added 50 new examples
dataset_v3.jsonl    # Fixed annotation errors in v2
```

**Golden rule: never overwrite a version.** If you need to update, create a new version file.

Git is also used for versioning datasets (especially on Hugging Face), but file naming is the simplest approach for small datasets.

A dataset card should record what changed between versions:
```python
dataset_card = {
    "version": "2.0",
    "changelog": "Added 50 adversarial examples. Fixed 3 mislabeled entries from v1."
}
```

## Your Turn — Exercise 1: Write and Read JSONL

Using `SAMPLE_DATASET`:

1. Create `jsonl_text` — the dataset serialized as a JSONL string (one JSON object per line)
2. Write `jsonl_text` to `Path('my_dataset_v1.jsonl')`
3. Read the file back and parse each line into a dict
4. Store the result in `loaded` (a list of 4 dicts)

Hint:
```python
jsonl_text = '\n'.join(json.dumps(entry) for entry in SAMPLE_DATASET)
Path('my_dataset_v1.jsonl').write_text(jsonl_text)
loaded = [json.loads(line) for line in Path('my_dataset_v1.jsonl').read_text().strip().split('\n')]
```

In [ ]:
# YOUR CODE HERE

# Step 1 & 2: Serialize to JSONL and write to file
jsonl_text = ""  # replace

# Step 3 & 4: Read back and parse
loaded = []  # replace

In [ ]:
check_length(loaded, 4, "loaded has 4 entries")
check_type(loaded[0], dict, "first loaded entry is a dict")
check_equal(loaded[0]['id'], 'ex_001', "first entry id is 'ex_001'")
print("\nLoaded entries:")
for entry in loaded:
    print(f"  [{entry['id']}] {entry['prompt']!r}")

## Your Turn — Exercise 2: Write and Read CSV

Using `SAMPLE_DATASET`:

1. Write it to `my_dataset.csv` using `csv.DictWriter` with `fieldnames=['id', 'prompt', 'expected_behavior', 'category']`
2. Read it back using `csv.DictReader`
3. Store the result as `csv_rows` (a list of dicts)

Hint:
```python
with open('my_dataset.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'prompt', 'expected_behavior', 'category'])
    writer.writeheader()
    writer.writerows(SAMPLE_DATASET)

with open('my_dataset.csv', newline='') as f:
    reader = csv.DictReader(f)
    csv_rows = list(reader)
```

In [ ]:
# YOUR CODE HERE

# Step 1: Write SAMPLE_DATASET to CSV

# Step 2 & 3: Read back with csv.DictReader
csv_rows = []  # replace

In [ ]:
check_length(csv_rows, 4, "csv_rows has 4 rows")
check_type(csv_rows[0], dict, "first csv row is a dict")
check_equal(csv_rows[0]['id'], 'ex_001', "first row id is 'ex_001'")
print("\nCSV rows:")
for row in csv_rows:
    print(f"  [{row['id']}] [{row['category']:8}] {row['prompt']!r}")

## Your Turn — Exercise 3: Write a Dataset Card

Create `dataset_card` — a dict describing `SAMPLE_DATASET`.

It must have exactly these keys:
- `name` (str): a short name for the dataset
- `version` (str): the version string, e.g. `"1.0"`
- `description` (str): a sentence or two describing what the dataset is
- `num_examples` (int): how many examples are in it (use `len(SAMPLE_DATASET)`)
- `categories` (list): a list of the category strings present in the dataset
- `created_by` (str): your name or identifier
- `limitations` (str): one sentence about what the dataset does NOT cover well

Fill in real values — this is practice for writing documentation that other researchers will rely on.

In [ ]:
# YOUR CODE HERE
# Write a dataset card dict with all required keys
dataset_card = {}  # replace this

In [ ]:
check_type(dataset_card, dict, "dataset_card is a dict")
check_keys(dataset_card, ['name', 'version', 'description', 'num_examples', 'categories', 'created_by', 'limitations'], "dataset_card has all required keys")
check_type(dataset_card['categories'], list, "categories is a list")
check_type(dataset_card['num_examples'], int, "num_examples is an int")

print("\nYour dataset card:")
for key, value in dataset_card.items():
    print(f"  {key}: {value!r}")

## Summary

- **JSONL**: one JSON object per line — the standard for eval datasets (streamable, appendable)
- **CSV**: tabular rows — good for analysis in pandas/spreadsheets, but no nested structures
- **JSON**: a full object/array — good for small datasets and config
- **Dataset cards** document what a dataset contains, who made it, and its limitations — treat them as required
- **Versioning**: use file naming (`v1`, `v2`, etc.) and never overwrite a previous version

**Next:** [04 — Mini Project](04_mini_project.ipynb)